<a href="https://colab.research.google.com/github/himanshucu01/fitsync-project-himanshu-bansal/blob/master/Module_2_Multibot_Chat_Playground_OpenRouter_Himanshu_Bansal.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 2 Graded Assignment — Multibot Chat Playground

**Student:** Himanshu Bansal  
**Course:** Automate with Gen AI & Agents  
**Provider:** OpenRouter

This notebook compares two LLMs using the **same prompts** across three tasks:
1. Creative Writing
2. Analytical Reasoning
3. Instruction Following

> **Security:** Enter your OpenRouter API key only when prompted. It is kept in the current Colab session and is **not written into this notebook**.


## 1. Install and Import Dependencies

In [ ]:
!pip -q install requests pandas

import requests
import time
import getpass
import pandas as pd
from IPython.display import display, Markdown


## 2. Configure OpenRouter Securely

Create an OpenRouter API key from your OpenRouter account before running this cell.

The key is requested at runtime and is not hard-coded into the notebook.


In [ ]:
OPENROUTER_API_KEY = getpass.getpass("Enter your OpenRouter API key: ").strip()

if not OPENROUTER_API_KEY:
    raise ValueError("No API key entered.")

OPENROUTER_URL = "https://openrouter.ai/api/v1/chat/completions"

HEADERS = {
    "Authorization": f"Bearer {OPENROUTER_API_KEY}",
    "Content-Type": "application/json",
    "HTTP-Referer": "https://colab.research.google.com/",
    "X-Title": "Module 2 Multibot Chat Playground"
}

print("API key received securely for this Colab session.")


## 3. Select Two Models

The model IDs below are examples available through OpenRouter. If a model is unavailable on your account, replace its ID with another currently available OpenRouter model.

The important requirement is that **the same prompt is sent to both models**.


In [ ]:
MODEL_A = "openai/gpt-4o-mini"
MODEL_B = "google/gemini-2.0-flash-001"

MODELS = {
    "Model A": MODEL_A,
    "Model B": MODEL_B
}

print("Selected models:")
for name, model in MODELS.items():
    print(f"{name}: {model}")


## 4. Reusable Model-Calling Function

This function sends requests to OpenRouter, measures response time, captures usage information when provided, and returns the generated answer.

Parameters such as `temperature`, `max_tokens`, and `model` influence generation behavior and output length.


In [ ]:
def call_model(model, prompt, temperature=0.7, max_tokens=600):
    payload = {
        "model": model,
        "messages": [
            {
                "role": "system",
                "content": "You are a helpful, accurate assistant. Follow the user's instructions exactly."
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        "temperature": temperature,
        "max_tokens": max_tokens
    }

    start = time.perf_counter()

    try:
        response = requests.post(
            OPENROUTER_URL,
            headers=HEADERS,
            json=payload,
            timeout=120
        )
        elapsed = time.perf_counter() - start

        response.raise_for_status()
        data = response.json()

        answer = data["choices"][0]["message"]["content"]
        usage = data.get("usage", {})

        return {
            "success": True,
            "model": model,
            "response": answer,
            "latency_seconds": round(elapsed, 3),
            "prompt_tokens": usage.get("prompt_tokens"),
            "completion_tokens": usage.get("completion_tokens"),
            "total_tokens": usage.get("total_tokens"),
            "error": ""
        }

    except Exception as e:
        elapsed = time.perf_counter() - start
        return {
            "success": False,
            "model": model,
            "response": "",
            "latency_seconds": round(elapsed, 3),
            "prompt_tokens": None,
            "completion_tokens": None,
            "total_tokens": None,
            "error": str(e)
        }


## 5. Define the Three Comparative Tasks

Each task uses one fixed prompt so that model differences are easier to attribute to the models rather than changing instructions.


In [ ]:
TASKS = {
    "Creative Writing": {
        "prompt": (
            "Write a 180-word short story titled 'The Last Signal'. "
            "The story must be set in a future where Earth has lost contact with every space colony. "
            "Use a suspenseful but hopeful tone. Include exactly one unexpected plot twist."
        ),
        "temperature": 0.9,
        "max_tokens": 350
    },

    "Analytical Reasoning": {
        "prompt": (
            "A college has 120 students. 70 study Python, 55 study Java, and 30 study both. "
            "How many students study at least one of the two languages? "
            "How many study neither? Show the calculation clearly and give the final answers."
        ),
        "temperature": 0.2,
        "max_tokens": 300
    },

    "Instruction Following": {
        "prompt": (
            "Return ONLY valid JSON with exactly these four keys: "
            ""title", "priority", "tasks", "deadline". "
            ""title" must be "Launch Website". "
            ""priority" must be one of "High", "Medium", or "Low". "
            ""tasks" must be an array containing exactly three strings. "
            ""deadline" must be "Friday". "
            "Do not use markdown, explanations, or code fences."
        ),
        "temperature": 0.1,
        "max_tokens": 250
    }
}


## 6. Run the Comparative Playtests

This cell sends each identical task prompt to both selected models and stores the outputs, timing, and token information.


In [ ]:
results = []

for task_name, task in TASKS.items():
    print(f"Running: {task_name}")

    for model_name, model_id in MODELS.items():
        result = call_model(
            model=model_id,
            prompt=task["prompt"],
            temperature=task["temperature"],
            max_tokens=task["max_tokens"]
        )

        result["task"] = task_name
        result["model_name"] = model_name
        results.append(result)

        print(f"  {model_name}: {result['latency_seconds']} seconds")

results_df = pd.DataFrame(results)

display(
    results_df[
        ["task", "model_name", "success", "latency_seconds",
         "prompt_tokens", "completion_tokens", "total_tokens", "error"]
    ]
)


## 7. Display Collected Outputs

Use these outputs as the evidence for the written reflection. Do not invent results; base your comparison on what the notebook actually returns.


In [ ]:
for task_name in TASKS:
    print("=" * 90)
    print(task_name.upper())

    subset = results_df[results_df["task"] == task_name]

    for _, row in subset.iterrows():
        print("\n" + "-" * 70)
        print(f"{row['model_name']} — {row['model'] if 'model' in row else MODELS[row['model_name']]}")
        print(f"Latency: {row['latency_seconds']} seconds")
        print(f"Total tokens: {row['total_tokens']}")
        print("\nOUTPUT:\n")
        print(row["response"])


## 8. Automatic Instruction-Following Check

The third task requests strict JSON. This cell checks whether each model returned parseable JSON and whether the required keys are present.


In [ ]:
import json

instruction_results = []

subset = results_df[results_df["task"] == "Instruction Following"]

required_keys = {"title", "priority", "tasks", "deadline"}

for _, row in subset.iterrows():
    text = row["response"].strip()

    try:
        parsed = json.loads(text)
        valid_json = True
        correct_keys = set(parsed.keys()) == required_keys
        correct_values = (
            parsed.get("title") == "Launch Website"
            and parsed.get("deadline") == "Friday"
            and parsed.get("priority") in {"High", "Medium", "Low"}
            and isinstance(parsed.get("tasks"), list)
            and len(parsed.get("tasks")) == 3
            and all(isinstance(x, str) for x in parsed["tasks"])
        )
    except Exception:
        valid_json = False
        correct_keys = False
        correct_values = False

    instruction_results.append({
        "Model": row["model_name"],
        "Valid JSON": valid_json,
        "Exact Keys": correct_keys,
        "Required Values": correct_values
    })

instruction_df = pd.DataFrame(instruction_results)
display(instruction_df)


## 9. Build a Model Comparison Summary

The scores below are intentionally left for human evaluation because quality, creativity, tone, and usefulness require judgment. Use the actual outputs above to assign a 1–5 score.

**Scoring guide:** 1 = poor, 3 = acceptable, 5 = excellent.


In [ ]:
comparison = []

for task_name in TASKS:
    for model_name in MODELS:
        comparison.append({
            "Task": task_name,
            "Model": model_name,
            "Quality / Usefulness (1-5)": "",
            "Creativity (1-5)": "",
            "Reasoning / Accuracy (1-5)": "",
            "Instruction Compliance (1-5)": "",
            "Tone (1-5)": "",
            "Reliability (1-5)": ""
        })

comparison_df = pd.DataFrame(comparison)
display(comparison_df)


## 10. Optional: Save the Experiment Results

Run this after completing your scoring. The CSV contains the collected experiment metadata and outputs and can be attached as supporting evidence if needed.


In [ ]:
results_df.to_csv("multibot_experiment_results.csv", index=False)
comparison_df.to_csv("multibot_comparison_scores.csv", index=False)

print("Saved:")
print("- multibot_experiment_results.csv")
print("- multibot_comparison_scores.csv")


# 11. Reflection Template — 700–900 Words

After running the notebook, write the final reflection using the actual evidence from your outputs.

## Suggested structure

### Introduction
Briefly explain the purpose of the Multibot Chat Playground, OpenRouter, the two selected models, and the three tasks.

### Creative Writing
Explain which model produced the stronger story. Compare originality, tone, structure, and whether the requested twist was effective.

### Analytical Reasoning
Compare the models' calculation, explanation, clarity, and consistency. Mention whether both reached the correct answer.

### Instruction Following
Compare strict JSON compliance. Discuss whether the models followed the exact key/value requirements and avoided additional text.

### API and Technical Factors
Discuss OpenRouter's common endpoint, model selection, system/user messages, temperature, max tokens, latency, and token usage. Explain how these factors can influence outputs.

### Automatic Model Routing
Propose a routing system that detects task type and selects a suitable model. For example:
- Creative tasks → model with strongest creativity
- Mathematical/analytical tasks → model with strongest accuracy
- Structured-output tasks → model with strongest format compliance
- Time-sensitive/simple tasks → faster, lower-cost model

### Conclusion
Summarize the strengths and limitations of using multiple models in one application.


## 12. Security Checklist

Before submitting:

- [ ] API key is entered with `getpass` and is not hard-coded.
- [ ] API key is not present in any output cell.
- [ ] API key is not committed to GitHub.
- [ ] Notebook runs from top to bottom without errors.
- [ ] At least two models were tested.
- [ ] All three tasks were tested with identical prompts across models.
- [ ] Actual outputs are included.
- [ ] Reflection is between 700 and 900 words.
- [ ] Comparison scores are based on actual results.
- [ ] GitHub notebook URL is accessible to the instructor.
